# Level-1 vs Level-2 anomaly detection comparison

This notebook reproduces the main comparison table for the study
`Price_vs_Orderbook_Data_for_Anomaly_Detection`.

**Runtime tip:** set the Colab runtime to GPU (`Runtime` → `Change runtime type`).
CPU works, but training takes longer.

Outputs are written under `results/`, including `table5_main_comparison.csv`.

## 1. Clone repository and install dependencies

In [ ]:
import os
from pathlib import Path

REPO = "Price_vs_Orderbook_Data_for_Anomaly_Detection"
REPO_URL = "https://github.com/OftGefragt/Price_vs_Orderbook_Data_for_Anomaly_Detection.git"

if not Path(REPO).exists():
    !git clone --depth 1 {REPO_URL}
else:
    print("Repository already present.")

%cd {REPO}
!pip -q install -r requirements-paper.txt
print("Working directory:", os.getcwd())

## 2. Run the unified comparison

The script trains each detector on BTC and scores ETH for both data levels,
then applies a shared top-quantile flagging rule (5% and 10%).

In [ ]:
# Optional: reduce Level-1 subsample sizes for a faster smoke test.
# Full-ish Colab run:
!python -m paper_eval.run_comparison --max-l1-train 12000 --max-l1-test 8000 --seed 42

# Faster smoke test (uncomment if needed):
# !python -m paper_eval.run_comparison --max-l1-train 3000 --max-l1-test 2000 --seed 42

## 3. Display Table 5

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from pathlib import Path

csv_path = Path("results/table5_main_comparison.csv")
md_path = Path("results/table5_main_comparison.md")

table = pd.read_csv(csv_path)
display(table)
if md_path.exists():
    display(Markdown(md_path.read_text()))

## 4. Download results from Colab

In [ ]:
from google.colab import files

files.download("results/table5_main_comparison.csv")
files.download("results/table5_main_comparison.md")

## Notes for the paper

1. Flag rates near 0.05 and 0.10 are expected by construction of the shared quantile rule.
   Use them to confirm calibration, then emphasize **cross-model Jaccard**, score dispersion,
   and runtime when interpreting Level-1 versus Level-2.
2. Level-1 rows may be subsampled for runtime. Report the subsample sizes used in the paper.
3. Deep models use short epochs so the notebook remains practical. Increase epochs in
   `paper_eval/models_anogan.py` and `paper_eval/models_gru_svdd.py` for final numbers if needed.